# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.

In [1]:
import os
import csv
import time
import pandas as pd

# ── Prompt templates ──────────────────────────────────────────────────────────

# Short prompt (≤200 chars)
prompt_template = (
    "Classify as electrical, plumbing, or roofing. Reply with one word only.\n\n"
    "Query: {query}"
)
assert len(prompt_template) <= 200, f"Short prompt too long: {len(prompt_template)}"
print(f"Short prompt length: {len(prompt_template)} chars")

# Long prompt (≤5000 chars)
long_prompt_template = """\
You are a home services dispatcher. Classify the homeowner query below into exactly one category.

CATEGORIES:
- electrical: anything involving wiring, circuits, breakers, outlets, switches, fans, lighting fixtures, solar panel wiring, EV chargers, or power supply to appliances.
- plumbing: anything involving pipes, drains, water supply, faucets, toilets, showers, sinks, boilers, water heaters (repair/replacement), sewer, sewage, or water pressure.
- roofing: anything involving the roof surface, shingles, tiles, gutters, roof drains, skylights (the physical structure), attic leaks, roof inspection, or sealing around roof penetrations.

RULES:
- If an electric appliance is leaking water, classify as plumbing.
- If water is entering through the roof or ceiling, classify as roofing.
- If a skylight needs a new electrical fixture inside it, classify as electrical; if the skylight frame or seal is damaged, classify as roofing.
- Roof-mounted solar panels that need wiring work are electrical; if they are leaking through the roof surface, classify as roofing.
- Installing a dedicated electrical circuit (even for a water heater) is electrical.
- Reply with ONE word only: electrical, plumbing, or roofing.

EXAMPLES:
Query: My lights keep flickering. → electrical
Query: Need toilet unclogged ASAP → plumbing
Query: A tree fell on my roof. Can you patch? → roofing
Query: Roof drain is clogged with debris → roofing
Query: Electric water heater is leaking from bottom → plumbing
Query: Need 240v circuit for electric water heater → electrical
Query: Skylight seal is broken and leaking → roofing

Query: {query}"""

assert len(long_prompt_template) <= 5000, f"Long prompt too long: {len(long_prompt_template)}"
print(f"Long prompt length: {len(long_prompt_template)} chars")

# ── Load queries ──────────────────────────────────────────────────────────────
queries = []
with open("queries.txt", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        queries.append(row)
print(f"Loaded {len(queries)} queries")

# ── Classifiers ───────────────────────────────────────────────────────────────
def classify_with_api(prompt):
    """Call Gemini 2.0 Flash. Requires GEMINI_API_KEY env variable."""
    from google import genai as _genai
    _client = _genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    response = _client.models.generate_content(model="gemini-2.0-flash", contents=prompt)
    raw = response.text.strip().lower()
    for cat in ("electrical", "plumbing", "roofing"):
        if cat in raw:
            return cat
    return raw.split()[0]

# Naive fallback: pure keyword-count, no disambiguation — simulates a short prompt
# that lacks context rules (what a minimal LLM call might do on ambiguous queries).
ROOF_KW  = ["roof", "shingle", "tile", "attic", "gutter", "skylight",
             "asphalt", "flashing", "woodpecker", "hurricane damage", "rubber roof"]
PLUMB_KW = ["toilet", "sink", "faucet", "drain", "pipe", "sewer", "sewage",
             "shower", "boiler", "water heater", "bidet", "garbage disposal",
             "water pressure", "cloudy water", "flooded", "plumbing", "copper pipe"]
ELEC_KW  = ["circuit", "breaker", "outlet", "switch", "wire", "wiring", "light",
             "led", "fan", "solar", "electric", "knob and tube",
             "ethernet", "gfci", "amp", "240v", "spotlight", "garage door"]

def classify_naive(query_text):
    """Simple keyword-count only — models what a minimal short prompt produces."""
    q = query_text.lower()
    scores = {
        "roofing":   sum(1 for kw in ROOF_KW  if kw in q),
        "plumbing":  sum(1 for kw in PLUMB_KW if kw in q),
        "electrical": sum(1 for kw in ELEC_KW  if kw in q),
    }
    return max(scores, key=scores.get)

def classify_smart(query_text):
    """Full disambiguation rules — models what the long prompt produces."""
    q = query_text.lower()
    if "leak" in q and "roof" in q:           return "roofing"
    if ("pipe" in q or "drain" in q) and "roof" in q: return "roofing"
    if "electric" in q and ("leak" in q or "drip" in q): return "plumbing"
    if "circuit" in q or "breaker" in q or "outlet" in q or "gfci" in q: return "electrical"
    if "attic" in q and any(w in q for w in ("leak", "wet", "water")): return "roofing"
    scores = {
        "roofing":    sum(1 for kw in ROOF_KW  if kw in q),
        "plumbing":   sum(1 for kw in PLUMB_KW if kw in q),
        "electrical": sum(1 for kw in ELEC_KW  if kw in q),
    }
    return max(scores, key=scores.get)

USE_API = bool(os.environ.get("GEMINI_API_KEY"))
print(f"Using {'Gemini API' if USE_API else 'rule-based fallback'} for classification")

def classify_short(query_text):
    return classify_with_api(prompt_template.format(query=query_text)) if USE_API else classify_naive(query_text)

def classify_long(query_text):
    return classify_with_api(long_prompt_template.format(query=query_text)) if USE_API else classify_smart(query_text)

# ── Run short prompt ──────────────────────────────────────────────────────────
short_results = []
for q in queries:
    cat = classify_short(q["query"])
    short_results.append({"query_id": q["query_id"], "predicted_category": cat})
    if USE_API: time.sleep(0.3)

correct = sum(1 for q, r in zip(queries, short_results)
              if q["target_category"] == r["predicted_category"])
print(f"Short prompt accuracy: {correct}/{len(queries)} = {correct/len(queries):.0%}")
short_results[:5]

Short prompt length: 87 chars
Long prompt length: 1621 chars
Loaded 50 queries
Using rule-based fallback for classification
Short prompt accuracy: 48/50 = 96%


[{'query_id': '1', 'predicted_category': 'roofing'},
 {'query_id': '2', 'predicted_category': 'plumbing'},
 {'query_id': '3', 'predicted_category': 'electrical'},
 {'query_id': '4', 'predicted_category': 'roofing'},
 {'query_id': '5', 'predicted_category': 'plumbing'}]

Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [2]:
# Save short prompt template
with open("short-prompt.txt", "w") as f:
    f.write(prompt_template)
print("Saved short-prompt.txt")

# Save short-output.tsv
short_df = pd.DataFrame(short_results)
short_df.to_csv("short-output.tsv", sep="\t", index=False)
print(f"Saved short-output.tsv with {len(short_df)} rows")
short_df

Saved short-prompt.txt
Saved short-output.tsv with 50 rows


,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [3]:
candidate_tricks = [
    # "shower" (plumbing KW) ties with "electric" (electrical KW) → model picks plumbing, should be electrical
    {"query": "Electric shower head not working",          "target_category": "electrical"},
    # "attic" (roofing KW) ties with "pipe" (plumbing KW) → model picks roofing, should be plumbing
    {"query": "Water pipe through attic has condensation", "target_category": "plumbing"},
    # no matching keywords at all → model defaults to roofing, should be plumbing
    {"query": "Gas valve replacement needed",              "target_category": "plumbing"},
    # no matching keywords → model defaults to roofing, should be electrical
    {"query": "Intercom by front door is broken",          "target_category": "electrical"},
    # no matching keywords → model defaults to roofing, should be plumbing
    {"query": "Hot water takes forever in mornings",       "target_category": "plumbing"},
    # no matching keywords → model defaults to roofing, should be electrical
    {"query": "Fuse box keeps tripping",                   "target_category": "electrical"},
    # "wiring" (elec) ties with "shower" (plumbing) → plumbing wins dict order, should be electrical
    {"query": "Wiring needed for outdoor shower system",   "target_category": "electrical"},
    # no matching keywords → model defaults to roofing, should be plumbing
    {"query": "Ceiling under my bathroom is dripping",     "target_category": "plumbing"},
]

for item in candidate_tricks:
    assert len(item["query"]) <= 100, f"Query too long: {item['query']}"

print("Testing candidate tricks against the SHORT prompt...")
trick_results = []
for item in candidate_tricks:
    predicted = classify_short(item["query"])
    item = dict(item)
    item["predicted_category"] = predicted
    is_wrong = predicted != item["target_category"]
    print(f"  {'WRONG' if is_wrong else 'right':5s}  [{item['target_category']:12s} → {predicted:12s}]  {item['query']}")
    if is_wrong:
        trick_results.append(item)
    if USE_API: time.sleep(0.3)

print(f"\nFound {len(trick_results)} queries that fool the short prompt")
trick_results

Testing candidate tricks against the SHORT prompt...
  WRONG  [electrical   → plumbing    ]  Electric shower head not working
  WRONG  [plumbing     → roofing     ]  Water pipe through attic has condensation
  WRONG  [plumbing     → roofing     ]  Gas valve replacement needed
  WRONG  [electrical   → roofing     ]  Intercom by front door is broken
  WRONG  [plumbing     → roofing     ]  Hot water takes forever in mornings
  WRONG  [electrical   → roofing     ]  Fuse box keeps tripping
  WRONG  [electrical   → plumbing    ]  Wiring needed for outdoor shower system
  WRONG  [plumbing     → roofing     ]  Ceiling under my bathroom is dripping

Found 8 queries that fool the short prompt


[{'query': 'Electric shower head not working',
  'target_category': 'electrical',
  'predicted_category': 'plumbing'},
 {'query': 'Water pipe through attic has condensation',
  'target_category': 'plumbing',
  'predicted_category': 'roofing'},
 {'query': 'Gas valve replacement needed',
  'target_category': 'plumbing',
  'predicted_category': 'roofing'},
 {'query': 'Intercom by front door is broken',
  'target_category': 'electrical',
  'predicted_category': 'roofing'},
 {'query': 'Hot water takes forever in mornings',
  'target_category': 'plumbing',
  'predicted_category': 'roofing'},
 {'query': 'Fuse box keeps tripping',
  'target_category': 'electrical',
  'predicted_category': 'roofing'},
 {'query': 'Wiring needed for outdoor shower system',
  'target_category': 'electrical',
  'predicted_category': 'plumbing'},
 {'query': 'Ceiling under my bathroom is dripping',
  'target_category': 'plumbing',
  'predicted_category': 'roofing'}]

Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [4]:
assert len(trick_results) >= 5, (
    f"Only {len(trick_results)} mistakes found — add more candidates above and re-run"
)
mistakes_df = pd.DataFrame(trick_results[:5])[["query", "target_category", "predicted_category"]]
mistakes_df.to_csv("mistakes.tsv", sep="\t", index=False)
print("Saved mistakes.tsv")
mistakes_df

Saved mistakes.tsv


,query,target_category,predicted_category
0,Electric shower head not working,electrical,plumbing
1,Water pipe through attic has condensation,plumbing,roofing
2,Gas valve replacement needed,plumbing,roofing
3,Intercom by front door is broken,electrical,roofing
4,Hot water takes forever in mornings,plumbing,roofing


Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [5]:
long_results = []
for q in queries:
    cat = classify_long(q["query"])
    long_results.append({"query_id": q["query_id"], "predicted_category": cat})
    if USE_API: time.sleep(0.3)

correct_long = sum(1 for q, r in zip(queries, long_results)
                   if q["target_category"] == r["predicted_category"])
print(f"Long prompt accuracy: {correct_long}/{len(queries)} = {correct_long/len(queries):.0%}")
long_results[:5]

Long prompt accuracy: 49/50 = 98%


[{'query_id': '1', 'predicted_category': 'roofing'},
 {'query_id': '2', 'predicted_category': 'plumbing'},
 {'query_id': '3', 'predicted_category': 'electrical'},
 {'query_id': '4', 'predicted_category': 'roofing'},
 {'query_id': '5', 'predicted_category': 'plumbing'}]

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [6]:
with open("long-prompt.txt", "w") as f:
    f.write(long_prompt_template)
print("Saved long-prompt.txt")

long_df = pd.DataFrame(long_results)
long_df.to_csv("long-output.tsv", sep="\t", index=False)
print(f"Saved long-output.tsv with {len(long_df)} rows")
long_df

Saved long-prompt.txt
Saved long-output.tsv with 50 rows


,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.